# Skyline Enrollment: Hypothesis Tests

Lesson 1.6 Guided Example. Runs t-tests on the Skyline Enrollment dataset,
demonstrates the multiple testing problem with simulation, and shows how
Bonferroni correction works.

In [17]:
import numpy as np
import pandas as pd
from itertools import combinations
from scipy import stats

In [4]:
skyline = pd.read_csv("../lesson-1-2-types-of-data/skyline_enrollments.csv")

python_for_beginners = skyline[skyline["course_name"] == "Python for Beginners"]["hours_studied"]
sql_basic = skyline[skyline["course_name"] == "SQL Basics"]["hours_studied"]

print(f"Python for Beginners: n={len(python_for_beginners)}, mean={python_for_beginners.mean():.2f} hours, std={python_for_beginners.std():.2f} hours")
print(f"SQL Basics: n={len(sql_basic)}, mean={sql_basic.mean():.2f} hours, std={sql_basic.std():.2f} hours")

Python for Beginners: n=24, mean=31.04 hours, std=14.11 hours
SQL Basics: n=24, mean=27.37 hours, std=15.03 hours


In [5]:
t_stat, p_value = stats.ttest_ind(python_for_beginners, sql_basic)

print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

t-statistic: 0.8729
p-value: 0.3873


In [6]:
mean_diff = python_for_beginners.mean() - sql_basic.mean()

# Pooled standard error of the difference
n1, n2 = len(python_for_beginners), len(sql_basic)
s1, s2 = python_for_beginners.std(ddof=1), sql_basic.std(ddof=1)
se_diff = np.sqrt(s1**2 / n1 + s2**2 / n2)

# 95% CI for the difference
margin = 1.96 * se_diff
ci_lower = mean_diff - margin
ci_upper = mean_diff + margin

print(f"Mean difference (Python for Beginners - SQL Basics): {mean_diff:.2f} hours")
print(f"95% CI for the difference: ({ci_lower:.2f}, {ci_upper:.2f}) hours")
print(f"p-value: {p_value:.4f}")

Mean difference (Python for Beginners - SQL Basics): 3.67 hours
95% CI for the difference: (-4.58, 11.92) hours
p-value: 0.3873


## Explanation

Students in Python for Beginner tend to have more hours studied on average 3.67 hours than students in SQL Basics, but this is not statistic detectable with a small sample size and huge variance between students (huge std). This could have easily happen by chance with p=0.387. It is hard to say the magnitude is practically meaningful when the plausible range runs from SQL may have 4.58 more hours studied to Python have 11.92 more hours studied than the sample average. 

In [19]:
courses = sorted(skyline["course_name"].unique())

rows = []
for a, b in combinations(sorted(skyline["course_name"].unique()), 2):
    x = skyline.loc[skyline["course_name"] == a, "hours_studied"]
    y = skyline.loc[skyline["course_name"] == b, "hours_studied"]
    _, p = stats.ttest_ind(x, y)
    rows.append({
        "course_a": a,
        "course_b": b,
        "mean_difference": round(x.mean() - y.mean(), 2),
        "p_value": round(p, 4),
    })

results = pd.DataFrame(rows)
print(results)

                    course_a              course_b  mean_difference  p_value
0  Introduction to Analytics  Python for Beginners            -5.51   0.2148
1  Introduction to Analytics            SQL Basics            -1.84   0.6906
2  Introduction to Analytics        Statistics 101            -1.26   0.7975
3  Introduction to Analytics  Tableau Fundamentals            -2.65   0.5703
4       Python for Beginners            SQL Basics             3.67   0.3873
5       Python for Beginners        Statistics 101             4.25   0.3912
6       Python for Beginners  Tableau Fundamentals             2.86   0.4996
7                 SQL Basics        Statistics 101             0.58   0.9105
8                 SQL Basics  Tableau Fundamentals            -0.81   0.8526
9             Statistics 101  Tableau Fundamentals            -1.39   0.7884


## Explanation

None of the results came with p < 0.05

The Bonferroni-corrected threshold for 10 tests at 0.05 family-wise error rate is 0.05/10 = 0.005

None the test reulst survive the Bonferroni-corrected threshold

With the 10 tests conducted, there were no false positive with the hypothesis test at 0.05 level. With no results coming back as "significant" it does not survive the Bonferroni correction. 